In [0]:
!pip install --upgrade --force-reinstall numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.1 MB/s eta 0:00:00
  Attempting uninstall: six
    Found existing installation: six 1.16.0
    Not uninstalling six at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3d8663c7-832d-4c63-9951-18f4df2280a9
    Can't uninstall 'six'. No files were found to uninstall.
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Not uninstalling numpy at /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3d8663c7-832d-4c63-9951-18f4df2280a9
    Can't uninstall 'numpy'. No files were found to uninstall.
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Not uninstalling python-dateutil at /databricks/python3/lib/python3.12/site-packages, outsid

In [0]:
%pip uninstall -y pandas numpy pandas-stubs pytz tzdata
%pip install "pandas==2.1.4" "numpy==1.26.4" "pytz<2025" "protobuf<5"

Found existing installation: pandas 3.0.3
Uninstalling pandas-3.0.3:
  Successfully uninstalled pandas-3.0.3
Found existing installation: numpy 2.4.6
Uninstalling numpy-2.4.6:
  Successfully uninstalled numpy-2.4.6
Found existing installation: pytz 2024.1
Not uninstalling pytz at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3d8663c7-832d-4c63-9951-18f4df2280a9
Can't uninstall 'pytz'. No files were found to uninstall.
Found existing installation: tzdata 2024.1
Not uninstalling tzdata at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3d8663c7-832d-4c63-9951-18f4df2280a9
Can't uninstall 'tzdata'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 97.3 MB/s eta 0:00:00
  Attempting uninstall: proto

In [0]:
# IMPORTANT: restart the Python process so the new wheels are actually used
dbutils.library.restartPython()

In [0]:
import os, re, json
import pandas as pd
import numpy as np
from difflib import SequenceMatcher

INBOUND_DIR = "/mnt/rxinvoices/aci_invoices"
OUTBOUND_DIR_DBFS = "/dbfs/mnt/rxinvoicesresults/"
CARRIER_NAME = "aci"
RATE_PATH = "/Workspace/Shared/PharmacyCarrier_Invoice_AI/data/rate_card.csv"

In [0]:
rate_df = pd.read_csv(RATE_PATH)
rate_df.columns = rate_df.columns.str.replace(" ", "", regex=False).str.upper()
rate_aci_df = rate_df[rate_df['COURIER'].str.contains('ASSOCIATED COURIERS LLC', case=False, na=False)].copy()
rate_aci_df['LINKED_PHARMA_CODE'] = rate_aci_df['LOCATION']

def clean_currency(column):
    return (column.astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
            .replace(['None', 'nan', 'NaN', '', '-', 'N/A', 'NA'], '0')
            .apply(lambda x: re.sub(r'[^0-9.\-]', '', x) if isinstance(x, str) else x)
            .replace('', '0')
            .astype(float))
    

cols_lst = ['ROUTEDCPM', 'ROUTEDCPS',
       'ROUTEMIN', 'ROUTEADDONFLATFEE', 'STATCPM', 'STATMIN', 'STAT-OTHER-INHOUSEDRIVERFEE',
       'WAITTIMECOST/MINUTE', 'VANCPM', 'VANSURCHARGE', 'VANSURCHARGEMIN',
       'MEDCARTCPM', 'MEDCARTADDITIONALCARTCHARGE', 'MEDCARTMIN',
       'BACKUPSTATCPM', 'BACKUPSTATMIN', 'LINEHAULCPM', 'SWEEPCPM',
       'SWEEPCPS', 'SWEEPCHARGEMINIMUM', 'AFTERHOURSSWEEPMIN',
       'OTHER-SORTINGCHARGE', 'OTHER-DRR-DRIVERRETENTION', 'OTHER-CYCLE']

for col in cols_lst:
    rate_aci_df[col] = clean_currency(rate_aci_df[col])

rate_aci_df

,KEY,OPPCFLAG,LOCATION,LOCATIONDESCRIPTION,PHARMACYNAME,CITY,STATE,COURIER,CONTRACTSTARTDATE,CONTRACTENDDATE,RATEEFFECTIVEDATE,ROUTEDCPM,ROUTEDCPS,ROUTEMIN,ROUTEADDONFLATFEE,STATCPM,STATMIN,STAT-OTHER-INHOUSEDRIVERFEE,WAITTIMECOST/MINUTE,VANCPM,VANSURCHARGE,VANSURCHARGEMIN,MEDCARTCPM,MEDCARTADDITIONALCARTCHARGE,MEDCARTMIN,BACKUPSTATCPM,BACKUPSTATMIN,LINEHAULCPM,SWEEPCPM,SWEEPCPS,SWEEPCHARGEMINIMUM,AFTERHOURSSWEEPMIN,OTHER-SORTINGCHARGE,OTHER-DRR-DRIVERRETENTION,OTHER-CYCLE,COMMENTS,LINKED_PHARMA_CODE
1,158060,0,15806,PAL VIRGINIA,CHRISTIANSBURG,CHRISTIANSBURG,VA,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,15806
17,326400,0,32640,7042 CHEMRX ALBANY OPERATIONS,ALBANY,ALBANY,NY,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.65,6.60,50.50,0.0,1.19,50.0,0.0,0.3,2.75,0.0,50.0,0.00,0.0,0.00,0.0,0.0,0.00,0.65,0.0,0.0,0.0,0.0,0.0,0.0,NaN,32640
18,326510,0,32651,7041 CHEMRX LONG ISLAND OPERATIONS,UNIONDALE,UNIONDALE,NY,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.80,6.50,65.00,0.0,1.25,65.0,0.0,0.3,2.50,0.0,65.0,0.00,0.0,0.00,0.0,0.0,0.00,0.80,0.0,0.0,0.0,0.0,0.0,0.0,NaN,32651
21,326850,0,32685,7143 MILLENNIUM COLUMBIA,COLUMBIA,COLUMBIA,MD,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.64,5.25,35.00,0.0,1.22,25.0,0.0,0.3,2.75,0.0,50.0,0.00,0.0,0.00,0.0,0.0,0.00,0.62,0.0,0.0,0.0,0.0,0.0,0.0,NaN,32685
22,329200,0,32920,7074 PMC ALBUQUERQUE,ALBUQUERQUE,ALBUQUERQUE,NM,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.60,5.80,50.00,0.0,1.23,50.0,0.0,0.3,2.75,0.0,50.0,0.00,0.0,0.00,0.0,0.0,0.00,0.60,0.0,0.0,0.0,0.0,0.0,0.0,NaN,32920
48,331100,0,33110,0927 PMC LOUISVILLE,LOUISVILLE,LOUISVILLE,KY,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.53,4.62,25.00,0.0,1.12,25.0,0.0,0.3,2.75,0.0,50.0,0.00,0.0,0.00,0.0,0.0,0.00,0.53,0.0,0.0,0.0,0.0,0.0,0.0,NaN,33110
65,331390,0,33139,7304 EXPRESS CARE LEESBURG,LEESBURG,LEESBURG,VA,ASSOCIATED COURIERS LLC,6/1/2025,3/31/2026,7/1/2025,0.78,2.51,46.34,40.0,1.25,45.0,0.0,0.3,0.00,0.0,0.0,0.75,0.0,0.75,0.0,0.0,0.55,0.78,0.0,0.0,0.0,0.0,0.0,0.0,NaN,33139
80,331630,0,33163,7303 EXPRESS CARE WASHINGTON,WASHINGTON,WASHINGTON,NC,ASSOCIATED COURIERS LLC,6/1/2025,3/31/2026,7/1/2025,0.71,2.54,45.76,40.0,1.25,40.0,0.0,0.3,0.00,0.0,0.0,0.75,0.0,0.75,0.0,0.0,0.55,0.68,0.0,0.0,0.0,0.0,0.0,0.0,NaN,33163
97,332980,0,33298,7358 WEST HENRIETTA,WEST HENRIETTA,WEST HENRIETTA,NY,ASSOCIATED COURIERS LLC,4/1/2022,3/31/2026,4/1/2022,0.63,5.25,50.00,0.0,1.16,50.0,0.0,0.3,2.75,0.0,50.0,0.00,0.0,0.00,0.0,0.0,0.00,0.63,0.0,0.0,0.0,0.0,0.0,0.0,NaN,33298


In [0]:
# Get only Excel files
files = [f for f in dbutils.fs.ls(INBOUND_DIR) if f.name.lower().endswith(".xlsx")]

# Sort by last modified time (descending) and take the latest file
latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]

fname = latest_file.name
file_slug = fname.replace(".xlsx", "").replace(" ", "_").strip("_")
RAW_PATH = latest_file.path.replace("dbfs:", "/dbfs")

print(f"--- Processing latest file: {fname} ---")

try:
    xlsx = pd.ExcelFile(RAW_PATH)
except Exception as e:
    raise Exception(f"Error reading file: {RAW_PATH}") from e

--- Processing latest file: Pharmerica Rochester-73935.xlsx ---


In [0]:
_sheet = next((s for s in xlsx.sheet_names if "detail" in s.lower()), xlsx.sheet_names[0])
raw_data_df = pd.read_excel(RAW_PATH, sheet_name=_sheet)
raw_data_df['LINKED_PHARMA_CODE'] = raw_data_df['PHARM_CODE']
raw_data_df

,COURIER_NAME,PHARM_CODE,PHARM_NAME,DESTIN_FACILITY_ID,DESTIN_NAME,DESTIN_st,DESTIN_CITY,DESTIN_STATE,DESTIN_POSTAL_CODE,ORDER_TIME,ORDER_DATE,PICKUP_TIME,PICKUP_DATE,DELIVERY_TIME,DELIVERY_DATE,PICKUP_FACILITY_ID,PICKUP_LOCATION,PICKUP_ADDRESS,PICKUP_CITY,PICKUP_STATE,PICKUP_ZIP,TOTAL_MILES,WAIT_MINUTES,TOLL_CHARGE,AMOUNT_CHARGES,DOCUTRACK_NO,ORDER_NUMBER,STAT_CODE,DELIVERY_TYPE,InvoiceNumber,InvoiceDate,RouteName,BillableStop,LINKED_PHARMA_CODE
0,Associated Couriers,33298,PharMerica-Rochester,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,3:00PM,2026-05-04,2:56PM,2026-05-04,2:56PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,0.0000,0,0.00,157.61,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),0,33298
1,Associated Couriers,33298,PharMerica-Rochester,183_HSTAT_7358,Margaret A. Stutzman Hstat,360 Forest Avenue,Buffalo,NY,14213,3:00PM,2026-05-04,2:56PM,2026-05-04,4:11PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,69.5060,0,4.75,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298
2,Associated Couriers,33298,PharMerica-Rochester,058_100_7358,Eden-West Seneca,3030 Clinton St,Buffalo,NY,14224,3:00PM,2026-05-04,2:56PM,2026-05-04,4:34PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,10.8665,0,0.00,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298
3,Associated Couriers,33298,PharMerica-Rochester,062_MEM_7358,Peregrine's Landing-Ckwg,575 Cayuga Creek Rd,Buffalo,NY,14227,3:00PM,2026-05-04,2:56PM,2026-05-04,4:48PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,2.5961,0,0.00,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298
4,Associated Couriers,33298,PharMerica-Rochester,061_ORCH_7358,Peregrine's Landing-OP,101 Sterling Dr,Orchard Park,NY,14127,3:00PM,2026-05-04,2:56PM,2026-05-04,5:05PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,8.0461,0,0.00,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
722,Associated Couriers,33298,PharMerica-Rochester,057_MED#2_7358,Eden Heights Of Olean,161 S 25th St,Olean,NY,14760,6:00PM,2026-05-09,4:25PM,2026-05-09,7:31PM,2026-05-09,PHAR7358,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,0.0000,0,0.00,NaN,NaN,May 9 2026 - 510686A013,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] WKND Hurlbut South 18:00 (SaSu),0,33298
723,Associated Couriers,33298,PharMerica-Rochester,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,6:00PM,2026-05-09,4:25PM,2026-05-09,7:31PM,2026-05-09,PHAR7358,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,95.7135,0,0.00,NaN,NaN,May 9 2026 - 510686A013,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] WKND Hurlbut South 18:00 (SaSu),0,33298
724,Associated Couriers,33298,PharMerica-Rochester,NaN,Eden Heights of Eden,4071 Hardt Rd,Eden,NY,14057,11:53AM,2026-05-04,11:56AM,2026-05-04,2:06PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,84.0000,0,0.00,103.19,1234.0,12144557,01 - Fac: Missed Cut & Need Before Next Delive...,STAT,73935,2026-05-11,NaN,1,33298
725,Associated Couriers,33298,PharMerica-Rochester,NaN,Alba De Vida,254 Virginia St,Buffalo,NY,14201,11:45AM,2026-05-06,10:58AM,2026-05-06,12:37PM,2026-05-06,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,70.0000,0,0.00,85.99,1234.0,12153365,20 - Other,STAT,73935,2026-05-11,NaN,1,33298


In [0]:
merge_col_lst = ['LINKED_PHARMA_CODE', 'ROUTEDCPM', 'ROUTEDCPS',
       'ROUTEMIN', 'ROUTEADDONFLATFEE', 'STATCPM', 'STATMIN', 'STAT-OTHER-INHOUSEDRIVERFEE',
       'WAITTIMECOST/MINUTE', 'VANCPM', 'VANSURCHARGE', 'VANSURCHARGEMIN',
       'MEDCARTCPM', 'MEDCARTADDITIONALCARTCHARGE', 'MEDCARTMIN',
       'BACKUPSTATCPM', 'BACKUPSTATMIN', 'LINEHAULCPM', 'SWEEPCPM',
       'SWEEPCPS', 'SWEEPCHARGEMINIMUM', 'AFTERHOURSSWEEPMIN',
       'OTHER-SORTINGCHARGE', 'OTHER-DRR-DRIVERRETENTION', 'OTHER-CYCLE']

raw_data_df_rate = raw_data_df.merge(rate_aci_df[merge_col_lst], on = 'LINKED_PHARMA_CODE', how = 'left')
raw_data_df_rate.head()

,COURIER_NAME,PHARM_CODE,PHARM_NAME,DESTIN_FACILITY_ID,DESTIN_NAME,DESTIN_st,DESTIN_CITY,DESTIN_STATE,DESTIN_POSTAL_CODE,ORDER_TIME,ORDER_DATE,PICKUP_TIME,PICKUP_DATE,DELIVERY_TIME,DELIVERY_DATE,PICKUP_FACILITY_ID,PICKUP_LOCATION,PICKUP_ADDRESS,PICKUP_CITY,PICKUP_STATE,PICKUP_ZIP,TOTAL_MILES,WAIT_MINUTES,TOLL_CHARGE,AMOUNT_CHARGES,DOCUTRACK_NO,ORDER_NUMBER,STAT_CODE,DELIVERY_TYPE,InvoiceNumber,InvoiceDate,RouteName,BillableStop,LINKED_PHARMA_CODE,ROUTEDCPM,ROUTEDCPS,ROUTEMIN,ROUTEADDONFLATFEE,STATCPM,STATMIN,STAT-OTHER-INHOUSEDRIVERFEE,WAITTIMECOST/MINUTE,VANCPM,VANSURCHARGE,VANSURCHARGEMIN,MEDCARTCPM,MEDCARTADDITIONALCARTCHARGE,MEDCARTMIN,BACKUPSTATCPM,BACKUPSTATMIN,LINEHAULCPM,SWEEPCPM,SWEEPCPS,SWEEPCHARGEMINIMUM,AFTERHOURSSWEEPMIN,OTHER-SORTINGCHARGE,OTHER-DRR-DRIVERRETENTION,OTHER-CYCLE
0,Associated Couriers,33298,PharMerica-Rochester,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,3:00PM,2026-05-04,2:56PM,2026-05-04,2:56PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,0.0000,0,0.00,157.61,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),0,33298,0.63,5.25,50.0,0.0,1.16,50.0,0.0,0.3,2.75,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.63,0.0,0.0,0.0,0.0,0.0,0.0
1,Associated Couriers,33298,PharMerica-Rochester,183_HSTAT_7358,Margaret A. Stutzman Hstat,360 Forest Avenue,Buffalo,NY,14213,3:00PM,2026-05-04,2:56PM,2026-05-04,4:11PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,69.5060,0,4.75,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298,0.63,5.25,50.0,0.0,1.16,50.0,0.0,0.3,2.75,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.63,0.0,0.0,0.0,0.0,0.0,0.0
2,Associated Couriers,33298,PharMerica-Rochester,058_100_7358,Eden-West Seneca,3030 Clinton St,Buffalo,NY,14224,3:00PM,2026-05-04,2:56PM,2026-05-04,4:34PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,10.8665,0,0.00,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298,0.63,5.25,50.0,0.0,1.16,50.0,0.0,0.3,2.75,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.63,0.0,0.0,0.0,0.0,0.0,0.0
3,Associated Couriers,33298,PharMerica-Rochester,062_MEM_7358,Peregrine's Landing-Ckwg,575 Cayuga Creek Rd,Buffalo,NY,14227,3:00PM,2026-05-04,2:56PM,2026-05-04,4:48PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,2.5961,0,0.00,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298,0.63,5.25,50.0,0.0,1.16,50.0,0.0,0.3,2.75,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.63,0.0,0.0,0.0,0.0,0.0,0.0
4,Associated Couriers,33298,PharMerica-Rochester,061_ORCH_7358,Peregrine's Landing-OP,101 Sterling Dr,Orchard Park,NY,14127,3:00PM,2026-05-04,2:56PM,2026-05-04,5:05PM,2026-05-04,NaN,PharMerica-Rochester,45 Becker Rd,West Henrietta,NY,14586,8.0461,0,0.00,NaN,NaN,May 4 2026 - 510686A029,NaN,ROUTED,73935,2026-05-11,[PMC-Rochester] Buffalo 15:00 (MTWRF),1,33298,0.63,5.25,50.0,0.0,1.16,50.0,0.0,0.3,2.75,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.63,0.0,0.0,0.0,0.0,0.0,0.0


## processing route data

In [ ]:
# --- Process route data: keep all original rows, place totals + rate-card calcs only on the AMOUNT_CHARGES anchor row ---
route_data_df_rate = raw_data_df_rate.copy()

# Make sure numeric cols are numeric
for c in ['TOTAL_MILES', 'BillableStop', 'TOLL_CHARGE', 'AMOUNT_CHARGES',
          'ROUTEDCPM', 'ROUTEDCPS', 'ROUTEMIN']:
    if c in route_data_df_rate.columns:
        route_data_df_rate[c] = pd.to_numeric(route_data_df_rate[c], errors='coerce')

# Aggregate ROUTED rows only -- STAT rows are not part of route totals
routed_mask = route_data_df_rate['DELIVERY_TYPE'] == 'ROUTED'
group_keys = ['ORDER_DATE', 'PHARM_CODE', 'RouteName']

agg = (
    route_data_df_rate.loc[routed_mask]
    .groupby(group_keys, as_index=False, dropna=False)
    .agg(TOTAL_MILES_TOTAL=('TOTAL_MILES', 'sum'),
         BillableStop_TOTAL=('BillableStop', 'sum'),
         TOLL_CHARGE_TOTAL=('TOLL_CHARGE', 'sum'))
)

# Identify the anchor row in each group: the leg that carries the invoiced AMOUNT_CHARGES
anchor_mask = routed_mask & route_data_df_rate['AMOUNT_CHARGES'].notna()

# Merge totals only onto anchor rows (left join on the anchor slice, then write back)
anchor_with_totals = (
    route_data_df_rate.loc[anchor_mask, group_keys]
    .merge(agg, on=group_keys, how='left')
)

# Initialize total/calc columns as NaN across the full frame
for c in ['TOTAL_MILES_TOTAL', 'BillableStop_TOTAL', 'TOLL_CHARGE_TOTAL',
          'ROUTED_BASE_COST', 'ROUTED_BASE_COST_MIN', 'total_cost', 'diff']:
    route_data_df_rate[c] = np.nan

# Write totals back onto the anchor rows
route_data_df_rate.loc[anchor_mask, 'TOTAL_MILES_TOTAL']   = anchor_with_totals['TOTAL_MILES_TOTAL'].values
route_data_df_rate.loc[anchor_mask, 'BillableStop_TOTAL']  = anchor_with_totals['BillableStop_TOTAL'].values
route_data_df_rate.loc[anchor_mask, 'TOLL_CHARGE_TOTAL']   = anchor_with_totals['TOLL_CHARGE_TOTAL'].values

# --- Rate-card calculations ONLY on the anchor row, using the aggregated totals ---
a = route_data_df_rate.loc[anchor_mask]  # view for readability

route_data_df_rate.loc[anchor_mask, 'ROUTED_BASE_COST'] = (
    a['TOTAL_MILES_TOTAL']  * a['ROUTEDCPM']
    + a['BillableStop_TOTAL'] * a['ROUTEDCPS']
)

# Apply route minimum
route_data_df_rate.loc[anchor_mask, 'ROUTED_BASE_COST_MIN'] = np.where(
    route_data_df_rate.loc[anchor_mask, 'ROUTED_BASE_COST'] < route_data_df_rate.loc[anchor_mask, 'ROUTEMIN'],
    route_data_df_rate.loc[anchor_mask, 'ROUTEMIN'],
    route_data_df_rate.loc[anchor_mask, 'ROUTED_BASE_COST']
)

# total_cost = floored base + summed tolls; diff vs invoiced AMOUNT_CHARGES
route_data_df_rate.loc[anchor_mask, 'total_cost'] = (
    route_data_df_rate.loc[anchor_mask, 'ROUTED_BASE_COST_MIN']
    + route_data_df_rate.loc[anchor_mask, 'TOLL_CHARGE_TOTAL'].fillna(0)
)
route_data_df_rate.loc[anchor_mask, 'diff'] = (
    route_data_df_rate.loc[anchor_mask, 'AMOUNT_CHARGES']
    - route_data_df_rate.loc[anchor_mask, 'total_cost'].round(2)
).round(2)

route_data_df_rate.head()


## save results

In [ ]:
outbound_dir = "/dbfs/mnt/rxinvoicesresults/" 
output_filename = os.path.splitext(fname)[0] + "_aci_results.csv" 
output_path = os.path.join(outbound_dir, output_filename) 
route_data_df_rate.to_csv(output_path, index=False) 
print(f"Extracted {len(route_data_df_rate)} rows. Saved to: {output_path}")